# Mission 3 ver4 — 전체 대화 + 보정된 앙상블

**이 노트북 하나만 Colab에 업로드하고 GPU 런타임에서 위에서부터 실행하세요.**
데이터는 `TL_서울_구급`, `VL_서울_구급` 폴더를 포함하도록 압축을 풉니다.
학습 코드·추론 코드가 아래에 모두 포함되어 있어 다른 `.py` 업로드는 필요 없습니다.

| 확인된 기존 결과 | Validation Macro F1 |
|---|---:|
| all_in_one_ver2 → results/mission3/ver1 | 0.600123 |
| mission3_ver3 → results/mission3/ver3 | 0.626015 |

ver4는 개선을 목표로 만든 새 실험입니다. **전체 GPU 학습 전에는 기존 버전보다 높은 점수를 보장할 수 없습니다.**
기존 384토큰 제한에서 Training의 11.27%가 잘렸습니다. ver4는 512토큰과 겹치는 window로 끝까지 읽습니다.

- `klue/roberta-large` 5개 fold 모델 + 문자/단어 TF-IDF Logistic Regression.
- Training의 90%에서 모델 학습·epoch 선택, 나머지 10%에서 **실제 최종 ensemble**의 가중치·임계값 보정.
- 보정 데이터 자체도 5분할해 보정 성능을 교차 평가합니다. Validation은 마지막 CSV 생성·평가에서만 사용합니다.
- 모델 입력은 양측 발화의 `text`만 사용하며 0개 증상 출력도 허용합니다.

A100/L4 같은 GPU 런타임과 충분한 Drive 공간을 준비하세요. 5개 large 가중치만 수 GB이며 학습 시간은 GPU에 따라 달라집니다.


In [ ]:
# 1. 필요한 버전을 설치합니다. Colab에 이미 있는 호환 torch는 유지합니다.
import subprocess
import sys
REQUIREMENTS = ['torch>=2.6,<3', 'transformers==4.57.1', 'numpy>=1.26,<3', 'scipy>=1.11,<2', 'scikit-learn>=1.4,<2', 'iterative-stratification==0.1.9', 'joblib>=1.3,<2']
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *REQUIREMENTS])


In [ ]:
# 2. 저장 위치. 경로를 직접 지정하려면 None을 실제 폴더 문자열로 바꾸세요.
import os
import tempfile
from pathlib import Path

START_DIR = Path.cwd().resolve()
TRAIN_LABEL_DIR = None
VALID_LABEL_DIR = None
try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount("/content/drive", force_remount=False)

candidates = [START_DIR, START_DIR / "DCC_AIdeas", *list(START_DIR.parents)[:4]]
PROJECT_ROOT = next((p for p in candidates if (p / "docs").is_dir() and (p / "src").is_dir()), START_DIR)
OUTPUT_DIR = (Path("/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver4") if IN_COLAB
              else PROJECT_ROOT / "results/mission3/ver4")
CODE_DIR = (Path("/content/mission3_ver4_runtime") if IN_COLAB
            else Path(tempfile.mkdtemp(prefix="mission3_ver4_runtime_")))
CODE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("결과 저장:", OUTPUT_DIR)
print("실행 코드:", CODE_DIR)


In [ ]:
# 3. /content를 먼저 찾고 없을 때 Drive를 찾습니다. Validation 없이 학습도 가능합니다.
def locate_labels(name, explicit=None, required=True):
    if explicit is not None:
        path = Path(explicit).expanduser().resolve()
        if not path.is_dir():
            raise FileNotFoundError(path)
        return path
    split = "Training" if name.startswith("TL_") else "Validation"
    short = "train" if name.startswith("TL_") else "validation"
    roots = ([Path("/content"), Path("/content/drive/MyDrive")] if IN_COLAB
             else [PROJECT_ROOT, START_DIR])
    for root in roots:
        candidates = [root / "mission3_data" / short / name, root / name]
        for base in [root, root / "DATA", root / "DCC_AIdeas/DATA", root / "mission3_data"]:
            candidates.extend([base / "대학부 데이터" / split / "2.라벨링데이터" / name,
                               base / split / "2.라벨링데이터" / name])
        for path in candidates:
            if path.is_dir():
                return path.resolve()
        if IN_COLAB:
            matches = []
            for current, directories, _ in os.walk(root):
                directories[:] = [d for d in directories if d not in
                                  {"drive", ".git", ".cache", ".config", "__pycache__", "node_modules", "sample_data"}]
                if Path(current).name == name:
                    matches.append(Path(current))
                    directories[:] = []
            if len(matches) > 1:
                raise ValueError(f"{name}가 여러 곳에 있습니다. 2번 셀에서 직접 지정하세요: {matches}")
            if matches:
                return matches[0].resolve()
    if required:
        raise FileNotFoundError(f"{name} 폴더를 못 찾았습니다. 2번 셀의 TRAIN_LABEL_DIR를 지정하세요.")
    return None

TRAIN_LABEL_DIR = locate_labels("TL_서울_구급", TRAIN_LABEL_DIR)
VALID_LABEL_DIR = locate_labels("VL_서울_구급", VALID_LABEL_DIR, required=False)
print("Training:", TRAIN_LABEL_DIR)
print("Validation:", VALID_LABEL_DIR)


## 4. 실행·제출 코드를 준비합니다

아래 `%%writefile` 셀은 노트북에 포함된 코드를 임시 실행 폴더에 저장합니다.
모델 학습은 뒤의 **6번 셀**에서 시작합니다. 코드 수정 후 노트북을 다시 만들 때는 저장소에서
`python src/mission3_ver4/build_notebook.py`를 실행할 수 있습니다.


In [ ]:
# writefile은 이 임시 폴더에만 기록합니다. 데이터/결과 경로는 절대경로로 보관했습니다.
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))


In [ ]:
%%writefile data.py
"""Mission 3: 모델 입력은 양측 발화의 text만, symptom은 학습/평가 정답만."""
from __future__ import annotations

import csv
import hashlib
import json
import unicodedata
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit
from sklearn.metrics import f1_score, precision_recall_fscore_support

TARGET_SYMPTOMS = (
    "고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란",
)


def dialogue_text(annotation: dict) -> str:
    """입력/정답과 무관하게 동일하게 전처리하며 발화 순서를 유지한다."""
    utterances = annotation.get("utterances")
    if not isinstance(utterances, list):
        utterances = annotation.get("utterences", [])
    if not isinstance(utterances, list):
        raise ValueError("utterances/utterences must be a list")
    texts = []
    for utterance in utterances:
        if isinstance(utterance, dict) and isinstance(utterance.get("text"), str):
            text = " ".join(unicodedata.normalize("NFC", utterance["text"]).split())
            if text:
                texts.append(text)
    return "\n".join(texts)


def target_vector(annotation: dict) -> list[int]:
    if "symptom" not in annotation:
        raise ValueError("Training/evaluation JSON has no symptom target")
    raw = annotation["symptom"]
    if raw is None:
        raw = []
    if isinstance(raw, str):
        raw = [raw]
    if not isinstance(raw, list) or any(not isinstance(x, str) for x in raw):
        raise ValueError("symptom must be a string or a list of strings")
    found = {x.strip() for x in raw}
    return [int(name in found) for name in TARGET_SYMPTOMS]


def text_group(text: str) -> str:
    # 파일명, 날짜, speaker 등은 모델 특징에도 중복 판단에도 사용하지 않는다.
    normalized = " ".join(unicodedata.normalize("NFC", text).split())
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


def load_dataset(label_dir: Path, include_targets: bool = False, workers: int = 8):
    paths = sorted(Path(label_dir).rglob("*.json"))
    if not paths:
        raise ValueError(f"JSON 파일이 없습니다: {label_dir}")
    if len({p.name for p in paths}) != len(paths):
        raise ValueError("동일한 JSON 파일명이 중복됩니다. 제출 CSV의 식별자가 모호합니다.")

    def read(path):
        with path.open(encoding="utf-8-sig") as handle:
            annotation = json.load(handle)
        if not isinstance(annotation, dict):
            raise ValueError(f"JSON object가 아닙니다: {path}")
        # 추론 경로에서는 target_vector를 호출하거나 symptom에 접근하지 않는다.
        return dialogue_text(annotation), target_vector(annotation) if include_targets else None

    with ThreadPoolExecutor(max_workers=workers) as executor:
        rows = list(executor.map(read, paths))
    texts = [row[0] for row in rows]
    targets = np.asarray([row[1] for row in rows], dtype=np.int8) if include_targets else None
    return paths, texts, targets


def grouped_targets(texts, targets):
    groups = np.asarray([text_group(text) for text in texts])
    unique, inverse = np.unique(groups, return_inverse=True)
    aggregate = np.zeros((len(unique), len(TARGET_SYMPTOMS)), dtype=np.int8)
    np.maximum.at(aggregate, inverse, targets)
    return groups, inverse, aggregate


def make_splits(texts, targets, calibration_size: float, n_folds: int, seed: int):
    """전체 Training의 보정 집합은 모든 모델 fit/epoch 선택에서 분리한다."""
    if not 0 < calibration_size < 0.5 or n_folds < 2:
        raise ValueError("calibration_size must be in (0, .5); n_folds >= 2")
    groups, inverse, aggregate = grouped_targets(texts, targets)
    group_ids = np.arange(len(aggregate))
    outer = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=calibration_size, random_state=seed)
    dev_groups, cal_groups = next(outer.split(group_ids, aggregate))
    development = np.flatnonzero(np.isin(inverse, dev_groups))
    calibration = np.flatnonzero(np.isin(inverse, cal_groups))
    splitter = MultilabelStratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed + 1)
    folds = []
    for train_g, stop_g in splitter.split(dev_groups, aggregate[dev_groups]):
        train_indices = np.flatnonzero(np.isin(inverse, dev_groups[train_g]))
        stop_indices = np.flatnonzero(np.isin(inverse, dev_groups[stop_g]))
        folds.append((train_indices, stop_indices))
    for name, indices in [("calibration", calibration), ("development", development),
                          *[(f"fold{i}/{part}", ids) for i, pair in enumerate(folds)
                            for part, ids in zip(("fit", "stop"), pair)]]:
        counts = targets[indices].sum(axis=0)
        if len(indices) == 0 or np.any(counts == 0) or np.any(counts == len(indices)):
            raise ValueError(f"{name}: 각 증상의 양성/음성이 필요합니다. 데이터/분할 수를 확인하세요.")
    return development, calibration, folds, groups


def write_json(path: Path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def write_predictions(paths, probabilities, thresholds, output: Path):
    probabilities = np.asarray(probabilities)
    if probabilities.shape != (len(paths), len(TARGET_SYMPTOMS)) or not np.isfinite(probabilities).all():
        raise ValueError("Invalid prediction shape or non-finite probabilities")
    predicted = probabilities >= np.asarray(thresholds)
    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["label file name", "symptom"])
        for path, flags in zip(paths, predicted):
            # 기존 제출 CSV와 같은 Python string-list 표현. 빈 list도 허용한다.
            writer.writerow([Path(path).name, str([s for s, flag in zip(TARGET_SYMPTOMS, flags) if flag])])
    return predicted.astype(np.int8)


def score_predictions(targets, predictions):
    precision, recall, f1, support = precision_recall_fscore_support(targets, predictions, zero_division=0)
    return {
        "macro_f1": float(f1_score(targets, predictions, average="macro", zero_division=0)),
        "samples": int(len(targets)),
        "per_symptom": {name: {"f1": float(f1[i]), "precision": float(precision[i]),
                                "recall": float(recall[i]), "support": int(support[i])}
                        for i, name in enumerate(TARGET_SYMPTOMS)},
    }


In [ ]:
%%writefile calibration.py
"""Training-only grouped calibration for nine independent symptom labels."""
from __future__ import annotations

import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.metrics import f1_score, precision_recall_fscore_support

if __package__:
    from .data import TARGET_SYMPTOMS
else:
    from data import TARGET_SYMPTOMS

WEIGHT_GRID = (0.0, 0.25, 0.5, 0.75, 1.0)
ALL_NEGATIVE_THRESHOLD = float(np.nextafter(1.0, np.inf))


def _probabilities(values, name):
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS):
        raise ValueError(f"{name} must have shape (samples, {len(TARGET_SYMPTOMS)})")
    if not len(values) or not np.isfinite(values).all():
        raise ValueError(f"{name} must be nonempty and finite")
    if np.any((values < 0) | (values > 1)):
        raise ValueError(f"{name} must contain probabilities in [0, 1]")
    return values


def _inputs(y_true, neural_probs, sparse_probs):
    neural = _probabilities(neural_probs, "neural_probs")
    sparse = _probabilities(sparse_probs, "sparse_probs")
    truth = np.asarray(y_true)
    if truth.shape != neural.shape or sparse.shape != neural.shape:
        raise ValueError("truth, neural_probs and sparse_probs must have identical shapes")
    if not np.isin(truth, (0, 1)).all():
        raise ValueError("y_true must contain only binary labels")
    return truth.astype(np.int8), neural, sparse


def _best_threshold(truth, probabilities):
    """Exact F1 maximum for >= decisions; tied probabilities stay together."""
    positives = int(truth.sum())
    if positives == 0:
        return ALL_NEGATIVE_THRESHOLD, 0.0
    if positives == len(truth):
        return 0.0, 1.0
    order = np.argsort(-probabilities, kind="stable")
    sorted_probs = probabilities[order]
    # A threshold can include a complete tied block, never part of one.
    boundaries = np.r_[np.flatnonzero(sorted_probs[:-1] != sorted_probs[1:]), len(truth) - 1]
    true_positives = np.cumsum(truth[order])[boundaries]
    scores = 2.0 * true_positives / (positives + boundaries + 1)
    thresholds = np.zeros(len(boundaries), dtype=np.float64)
    has_next = boundaries < len(truth) - 1
    upper = sorted_probs[boundaries[has_next]]
    lower = sorted_probs[boundaries[has_next] + 1]
    midpoint = lower + (upper - lower) / 2
    # Adjacent floating-point values may have no representable midpoint.
    thresholds[has_next] = np.where(midpoint > lower, midpoint, upper)
    best = np.flatnonzero(scores == scores.max())
    # An explicit deterministic tie break preserves the exact maximum.
    winner = min(best, key=lambda i: (abs(thresholds[i] - 0.5), -thresholds[i]))
    return float(thresholds[winner]), float(scores[winner])


def fit_calibrator(y_true, neural_probs, sparse_probs):
    """Select each label's blend and threshold using supplied training labels."""
    truth, neural, sparse = _inputs(y_true, neural_probs, sparse_probs)
    weights, thresholds = [], []
    for column in range(len(TARGET_SYMPTOMS)):
        candidates = []
        for weight in WEIGHT_GRID:
            blended = weight * neural[:, column] + (1.0 - weight) * sparse[:, column]
            threshold, score = _best_threshold(truth[:, column], blended)
            candidates.append((score, -abs(weight - 0.5), weight, threshold))
        _, _, weight, threshold = max(candidates)
        weights.append(float(weight))
        thresholds.append(float(threshold))
    return {"neural_weights": weights, "thresholds": thresholds}


def apply_calibrator(neural_probs, sparse_probs, calibrator):
    """Return blended probabilities; use >= stored thresholds for labels."""
    neural = _probabilities(neural_probs, "neural_probs")
    sparse = _probabilities(sparse_probs, "sparse_probs")
    if neural.shape != sparse.shape:
        raise ValueError("neural_probs and sparse_probs must have identical shapes")
    weights = np.asarray(calibrator["neural_weights"], dtype=np.float64)
    thresholds = np.asarray(calibrator["thresholds"], dtype=np.float64)
    expected = (len(TARGET_SYMPTOMS),)
    if weights.shape != expected or thresholds.shape != expected:
        raise ValueError("calibrator must contain one weight and threshold per symptom")
    if not np.isfinite(weights).all() or np.any((weights < 0) | (weights > 1)):
        raise ValueError("calibrator weights must be finite and in [0, 1]")
    if not np.isfinite(thresholds).all() or np.any((thresholds < 0) | (thresholds > ALL_NEGATIVE_THRESHOLD)):
        raise ValueError("calibrator thresholds must be finite and in [0, nextafter(1, inf)]")
    return weights * neural + (1.0 - weights) * sparse


def _grouped_splits(truth, groups, seed, n_splits):
    groups = np.asarray(groups)
    if groups.ndim != 1 or len(groups) != len(truth):
        raise ValueError("groups must contain one group ID per sample")
    if not isinstance(n_splits, (int, np.integer)) or n_splits < 2:
        raise ValueError("n_splits must be an integer >= 2")
    unique_groups, inverse = np.unique(groups, return_inverse=True)
    if len(unique_groups) < 2:
        raise ValueError("cross-fit calibration requires at least two distinct groups")
    n_splits = min(n_splits, len(unique_groups))
    group_targets = np.zeros((len(unique_groups), truth.shape[1]), dtype=np.int8)
    np.maximum.at(group_targets, inverse, truth)
    splitter = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for train_groups, test_groups in splitter.split(np.zeros(len(unique_groups)), group_targets):
        train = np.flatnonzero(np.isin(inverse, train_groups))
        test = np.flatnonzero(np.isin(inverse, test_groups))
        if not len(train) or not len(test):
            raise ValueError("grouped calibration produced an empty fold")
        yield train, test


def calibrate_crossfit(y_true, neural_probs, sparse_probs, groups, seed=42, n_splits=5):
    """Cross-fit within Training calibration data, then fit deployment settings.

    Base-model predictions must themselves come from samples excluded from
    base-model training and model selection. Never pass official Validation.
    """
    truth, neural, sparse = _inputs(y_true, neural_probs, sparse_probs)
    crossfit_predictions = np.zeros_like(truth)
    seen = np.zeros(len(truth), dtype=np.int8)
    fold_counts = []
    for train, test in _grouped_splits(truth, groups, seed, n_splits):
        calibrator = fit_calibrator(truth[train], neural[train], sparse[train])
        probabilities = apply_calibrator(neural[test], sparse[test], calibrator)
        crossfit_predictions[test] = probabilities >= np.asarray(calibrator["thresholds"])
        seen[test] += 1
        fold_counts.append({"fit_samples": int(len(train)), "score_samples": int(len(test))})
    if not np.all(seen == 1):
        raise RuntimeError("each calibration sample must be scored exactly once")
    final_calibrator = fit_calibrator(truth, neural, sparse)
    fitted = apply_calibrator(neural, sparse, final_calibrator) >= np.asarray(final_calibrator["thresholds"])
    precision, recall, f1, support = precision_recall_fscore_support(
        truth, crossfit_predictions, average=None, zero_division=0
    )
    report = {
        "score_scope": "Training-only calibration; cross-fit estimates calibration generalization, not official Validation.",
        "calibration_fit_macro_f1": float(f1_score(truth, fitted, average="macro", zero_division=0)),
        "calibration_crossfit_macro_f1": float(f1.mean()),
        "per_symptom_crossfit": {
            symptom: {"precision": float(p), "recall": float(r), "f1": float(f), "support": int(s)}
            for symptom, p, r, f, s in zip(TARGET_SYMPTOMS, precision, recall, f1, support)
        },
        "calibration_samples": int(len(truth)),
        "calibration_groups": int(len(np.unique(groups))),
        "crossfit_splits": len(fold_counts),
        "crossfit_fold_counts": fold_counts,
    }
    return final_calibrator, report


In [ ]:
%%writefile neural.py
"""Document-level multi-label transformer; every dialogue window is retained."""

from __future__ import annotations

import math
import os
import random
import re
import tempfile
import time
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import average_precision_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoConfig, AutoModel, get_linear_schedule_with_warmup


NUM_LABELS = 9


def tokenize_dialogues(texts, tokenizer, max_length=512, stride=64):
    """Return documents of token windows, including the last window of long calls.

    Only ordinary tokenizer special tokens are used. The stride is an overlap in
    tokens, not an upper bound on the number of windows or dialogue length.
    """
    usable_length = max_length - tokenizer.num_special_tokens_to_add(pair=False)
    if usable_length < 1 or not 0 <= stride < usable_length:
        raise ValueError("max_length must fit special tokens; 0 <= stride < usable length")
    texts = list(texts)
    documents = [[] for _ in texts]
    for start in range(0, len(texts), 128):
        encoded = tokenizer(
            texts[start:start + 128], truncation=True, max_length=max_length,
            stride=stride, return_overflowing_tokens=True, padding=False,
            return_attention_mask=True,
        )
        if "overflow_to_sample_mapping" not in encoded:
            raise ValueError("Use a fast tokenizer (AutoTokenizer(..., use_fast=True))")
        for window, sample in enumerate(encoded["overflow_to_sample_mapping"]):
            documents[start + int(sample)].append({
                key: encoded[key][window]
                for key in tokenizer.model_input_names if key in encoded
            })
    if any(not windows for windows in documents):
        raise ValueError("Tokenizer returned a document without any windows")
    return documents


class DialogueDataset(Dataset):
    def __init__(self, documents, targets=None):
        self.documents, self.targets = documents, targets

    def __len__(self):
        return len(self.documents)

    def __getitem__(self, index):
        item = {"windows": self.documents[index]}
        if self.targets is not None:
            item["labels"] = self.targets[index]
        return item


class DialogueCollator:
    """Flatten windows for the encoder while keeping exactly one target per call."""

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, examples):
        windows, doc_indices = [], []
        for index, example in enumerate(examples):
            if not example["windows"]:
                raise ValueError("A document must contain at least one token window")
            windows.extend(example["windows"])
            doc_indices.extend([index] * len(example["windows"]))
        batch = dict(self.tokenizer.pad(windows, padding=True, return_tensors="pt"))
        batch["doc_indices"] = torch.tensor(doc_indices, dtype=torch.long)
        if "labels" in examples[0]:
            batch["labels"] = torch.from_numpy(np.asarray(
                [example["labels"] for example in examples], dtype=np.float32,
            ))
        return batch


class DialogueClassifier(nn.Module):
    """Label-specific attention pools complete dialogues before computing BCE."""

    def __init__(self, encoder, num_labels=NUM_LABELS, dropout=0.2,
                 window_batch_size=8):
        super().__init__()
        self.encoder = encoder
        self.window_batch_size = int(window_batch_size)
        if self.window_batch_size < 1:
            raise ValueError("window_batch_size must be positive")
        hidden = encoder.config.hidden_size
        self.projection = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.Tanh())
        self.dropout = nn.Dropout(dropout)
        self.attention_queries = nn.Parameter(torch.empty(num_labels, hidden))
        self.label_weights = nn.Parameter(torch.empty(num_labels, hidden))
        self.label_bias = nn.Parameter(torch.zeros(num_labels))
        nn.init.normal_(self.attention_queries, std=0.02)
        nn.init.normal_(self.label_weights, std=0.02)

    def forward(self, doc_indices, **inputs):
        representations = []
        for start in range(0, len(doc_indices), self.window_batch_size):
            window_inputs = {key: value[start:start + self.window_batch_size]
                             for key, value in inputs.items()}
            hidden = self.encoder(**window_inputs).last_hidden_state
            mask = window_inputs["attention_mask"].unsqueeze(-1).to(hidden.dtype)
            mean = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
            representations.append(self.projection(torch.cat([hidden[:, 0], mean], -1)))
        windows = self.dropout(torch.cat(representations))
        logits = []
        # Each label learns which part of a potentially long call provides evidence.
        for document_index in range(int(doc_indices.max().item()) + 1):
            selected = windows[doc_indices == document_index]
            scores = selected @ self.attention_queries.T / math.sqrt(selected.shape[-1])
            attention = torch.softmax(scores.float(), dim=0).to(selected.dtype)
            pooled = attention.T @ selected
            logits.append((self.dropout(pooled) * self.label_weights).sum(-1) + self.label_bias)
        return torch.stack(logits)


def _device(config):
    device = torch.device(config.get("device", "cuda" if torch.cuda.is_available() else "cpu"))
    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is unavailable. Select a GPU runtime.")
    return device


def _amp(device):
    dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
    return torch.amp.autocast(device_type=device.type, dtype=dtype, enabled=device.type == "cuda")


def _loader(documents, targets, tokenizer, batch_size, shuffle=False, seed=0):
    if int(batch_size) < 1:
        raise ValueError("batch_size must be positive")
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        DialogueDataset(documents, targets), batch_size=int(batch_size),
        shuffle=shuffle, collate_fn=DialogueCollator(tokenizer),
        num_workers=0, generator=generator,
    )


@torch.inference_mode()
def _predict(model, loader, device):
    model.eval()
    probabilities = []
    for batch in loader:
        batch.pop("labels", None)
        batch = {key: value.to(device) for key, value in batch.items()}
        with _amp(device):
            logits = model(**batch)
        probabilities.append(logits.float().sigmoid().cpu().numpy())
    return np.concatenate(probabilities) if probabilities else np.empty((0, NUM_LABELS), dtype=np.float32)


def _optimizer(model, config):
    depth = int(getattr(model.encoder.config, "num_hidden_layers", 0))
    decay = float(config.get("layerwise_lr_decay", 0.9))
    if not 0 < decay <= 1:
        raise ValueError("layerwise_lr_decay must be in (0, 1]")
    groups = {}
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.startswith("encoder."):
            match = re.search(r"\.layer\.(\d+)\.", name)
            level = int(match.group(1)) + 1 if match else (0 if ".embeddings." in name else depth + 1)
            lr = float(config.get("learning_rate", 1.5e-5)) * decay ** (depth + 1 - level)
        else:
            lr = float(config.get("head_learning_rate", 1e-4))
        wd = 0.0 if parameter.ndim < 2 or name.endswith("bias") else float(config.get("weight_decay", 0.01))
        groups.setdefault((lr, wd), []).append(parameter)
    return torch.optim.AdamW([{"params": params, "lr": lr, "weight_decay": wd}
                             for (lr, wd), params in groups.items()])


def _atomic_save(checkpoint, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary = tempfile.mkstemp(prefix=path.name + ".", suffix=".tmp", dir=path.parent)
    os.close(descriptor)
    try:
        torch.save(checkpoint, temporary)
        os.replace(temporary, path)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)


def _targets(targets, documents, name):
    targets = np.asarray(targets, dtype=np.float32)
    if targets.shape != (len(documents), NUM_LABELS) or not len(documents):
        raise ValueError(f"{name} requires nonempty documents and targets of shape (N, {NUM_LABELS})")
    if not np.isin(targets, [0, 1]).all():
        raise ValueError(f"{name} targets must contain only 0 and 1")
    return targets


def train_model(train_documents, train_targets, stop_documents, stop_targets,
                tokenizer, config: dict, checkpoint_path: Path, seed: int):
    """Train on documents; select best epoch only by stopping-set macro AP.

    This function does not tune thresholds and never uses calibration targets.
    GPU enforcement for production belongs to the orchestration entry point.
    """
    train_targets = _targets(train_targets, train_documents, "train")
    stop_targets = _targets(stop_targets, stop_documents, "stop")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    device = _device(config)
    max_epochs, patience = int(config.get("max_epochs", 8)), int(config.get("patience", 2))
    accumulation = int(config.get("gradient_accumulation_steps", 4))
    if min(max_epochs, patience, accumulation) < 1:
        raise ValueError("max_epochs, patience and gradient_accumulation_steps must be positive")
    model_name = config.get("model_name", "klue/roberta-large")
    encoder = AutoModel.from_pretrained(model_name)
    if config.get("gradient_checkpointing", True):
        encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    if hasattr(encoder.config, "use_cache"):
        encoder.config.use_cache = False
    architecture = {"num_labels": NUM_LABELS, "dropout": float(config.get("dropout", 0.2)),
                    "window_batch_size": int(config.get("window_batch_size", 8))}
    model = DialogueClassifier(encoder, **architecture).to(device)
    train_loader = _loader(train_documents, train_targets, tokenizer,
                           config.get("batch_size", 2), shuffle=True, seed=seed)
    stop_loader = _loader(stop_documents, None, tokenizer, config.get("inference_batch_size", 4))
    positives = train_targets.sum(axis=0)
    if (positives == 0).any():
        raise ValueError(f"Training split has no positive examples for labels {np.flatnonzero(positives == 0).tolist()}")
    pos_weight = np.clip(((len(train_targets) - positives) / positives).clip(min=1)
                         ** float(config.get("pos_weight_power", 0.5)), 1, 5)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.as_tensor(pos_weight, device=device))
    optimizer = _optimizer(model, config)
    total_steps = math.ceil(len(train_loader) / accumulation) * max_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * float(config.get("warmup_ratio", 0.1))),
        num_training_steps=total_steps,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and not torch.cuda.is_bf16_supported())
    valid_labels = stop_targets.sum(axis=0) > 0
    if not valid_labels.any():
        raise ValueError("Stopping split has no positive labels; macro AP cannot select an epoch")
    if not valid_labels.all():
        print(f"Warning: stop AP excludes labels without positives: {np.flatnonzero(~valid_labels).tolist()}", flush=True)
    best_ap, best_epoch, stale, history = -1.0, 0, 0, []
    for epoch in range(1, max_epochs + 1):
        epoch_started = time.monotonic()
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss_sum, document_count = 0.0, 0
        for step, batch in enumerate(train_loader):
            labels = batch.pop("labels").to(device)
            batch = {key: value.to(device) for key, value in batch.items()}
            # Match a single full document batch, including a shorter last batch.
            group_start = (step // accumulation) * accumulation
            group_documents = min(accumulation * train_loader.batch_size,
                                  len(train_documents) - group_start * train_loader.batch_size)
            with _amp(device):
                loss = criterion(model(**batch).float(), labels)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite training loss at epoch={epoch}, batch={step}")
            scaler.scale(loss * len(labels) / group_documents).backward()
            loss_sum += loss.detach().item() * len(labels)
            document_count += len(labels)
            if (step + 1) % accumulation == 0 or step + 1 == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= old_scale:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            log_every = int(config.get("log_every", 100))
            if log_every > 0 and (step + 1) % log_every == 0:
                elapsed = (time.monotonic() - epoch_started) / 60
                print(f"seed={seed} epoch={epoch} batch={step + 1}/{len(train_loader)} "
                      f"loss={loss_sum / document_count:.5f} minutes={elapsed:.1f}", flush=True)
        probabilities = _predict(model, stop_loader, device)
        if not np.isfinite(probabilities).all():
            raise FloatingPointError("Stopping predictions contain non-finite values")
        score = float(average_precision_score(stop_targets[:, valid_labels],
                                              probabilities[:, valid_labels], average="macro"))
        row = {"epoch": epoch, "train_loss": loss_sum / document_count,
               "stop_macro_ap": score, "stop_labels_with_positives": int(valid_labels.sum()),
               "minutes": (time.monotonic() - epoch_started) / 60}
        history.append(row)
        print(f"seed={seed} epoch={epoch}/{max_epochs} loss={row['train_loss']:.5f} "
              f"stop_AP={score:.5f} minutes={row['minutes']:.1f}", flush=True)
        if score > best_ap + 1e-6:
            best_ap, best_epoch, stale = score, epoch, 0
            _atomic_save({
                "format_version": 1, "encoder_config": encoder.config.to_dict(),
                "architecture": architecture, "model_name": str(model_name),
                "model_state": {key: value.detach().cpu() for key, value in model.state_dict().items()},
                "epoch": epoch, "stop_ap": score, "seed": int(seed),
            }, checkpoint_path)
        else:
            stale += 1
            if stale >= patience:
                break
    return {"best_epoch": best_epoch, "best_stop_ap": best_ap, "history": history}


def predict_checkpoint(checkpoint_path, documents, tokenizer, config):
    """Rebuild from the saved config and weights; no pretrained network fetch."""
    device = _device(config)
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    if checkpoint.get("format_version") != 1:
        raise ValueError(f"Unsupported neural checkpoint format: {checkpoint_path}")
    encoder_data = dict(checkpoint["encoder_config"])
    model_type = encoder_data.pop("model_type")
    encoder_config = AutoConfig.for_model(model_type, **encoder_data)
    architecture = dict(checkpoint["architecture"])
    architecture["window_batch_size"] = int(config.get("window_batch_size", architecture["window_batch_size"]))
    model = DialogueClassifier(AutoModel.from_config(encoder_config), **architecture)
    model.load_state_dict(checkpoint["model_state"], strict=True)
    del checkpoint
    model.to(device)
    loader = _loader(documents, None, tokenizer, config.get("inference_batch_size", 4))
    predictions = _predict(model, loader, device)
    if not np.isfinite(predictions).all():
        raise FloatingPointError("Inference predictions contain non-finite values")
    return predictions


In [ ]:
%%writefile pipeline.py
"""Training-only model selection, ensemble calibration, and portable inference."""
from __future__ import annotations

import ast
import csv
import gc
import hashlib
import importlib.metadata
import json
import shutil
from pathlib import Path

import joblib
import numpy as np
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion
from transformers import AutoTokenizer

if __package__:
    from .calibration import apply_calibrator, calibrate_crossfit
    from .data import TARGET_SYMPTOMS, load_dataset, make_splits, score_predictions, write_json, write_predictions
    from .neural import predict_checkpoint, tokenize_dialogues, train_model
else:
    from calibration import apply_calibrator, calibrate_crossfit
    from data import TARGET_SYMPTOMS, load_dataset, make_splits, score_predictions, write_json, write_predictions
    from neural import predict_checkpoint, tokenize_dialogues, train_model


DEFAULT_CONFIG = {
    "model_name": "klue/roberta-large",
    "seed": 42,
    "n_folds": 5,
    "calibration_size": 0.10,
    "calibration_folds": 5,
    "max_length": 512,
    "stride": 64,
    "max_epochs": 6,
    "patience": 2,
    "batch_size": 2,
    "inference_batch_size": 8,
    "window_batch_size": 2,
    "gradient_accumulation_steps": 16,
    "learning_rate": 1.5e-5,
    "head_learning_rate": 1e-4,
    "layerwise_lr_decay": 0.95,
    "weight_decay": 0.01,
    "warmup_ratio": 0.10,
    "dropout": 0.20,
    "pos_weight_power": 0.5,
    "gradient_checkpointing": True,
    "device": "cuda",
    "require_cuda": True,
    "char_features": 120_000,
    "word_features": 40_000,
    "min_df": 2,
    "logistic_c": 4.0,
}
RUNTIME_FILES = ("data.py", "calibration.py", "neural.py", "pipeline.py", "train.py", "inference.py", "evaluate.py")


def fit_sparse(texts, targets, config):
    """문자 표현과 단어 순서를 보완하는, Training에만 fit한 TF-IDF 모델."""
    features = FeatureUnion([
        ("char", TfidfVectorizer(analyzer="char", ngram_range=(2, 5),
                                max_features=config["char_features"], min_df=config["min_df"],
                                sublinear_tf=True, dtype=np.float32)),
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), token_pattern=r"(?u)\b\w+\b",
                                max_features=config["word_features"], min_df=config["min_df"],
                                sublinear_tf=True, dtype=np.float32)),
    ])
    matrix = features.fit_transform(texts)
    classifiers = []
    for column, name in enumerate(TARGET_SYMPTOMS):
        truth = targets[:, column]
        if len(np.unique(truth)) == 1:
            classifier = float(truth[0])
        else:
            classifier = LogisticRegression(C=config["logistic_c"], class_weight="balanced",
                                            solver="liblinear", max_iter=1500, random_state=config["seed"])
            classifier.fit(matrix, truth)
        classifiers.append(classifier)
        print(f"TF-IDF fit: {name}", flush=True)
    return {"features": features, "classifiers": classifiers}


def predict_sparse(model, texts):
    if not texts:
        return np.empty((0, len(TARGET_SYMPTOMS)), dtype=np.float32)
    # 큰 Test 폴더도 전체 sparse matrix를 RAM에 동시에 올리지 않는다.
    batches = []
    for start in range(0, len(texts), 1024):
        matrix = model["features"].transform(texts[start:start + 1024])
        columns = [np.full(matrix.shape[0], clf) if isinstance(clf, float)
                   else clf.predict_proba(matrix)[:, 1] for clf in model["classifiers"]]
        batches.append(np.column_stack(columns))
    return np.concatenate(batches).astype(np.float32)


def _release_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _torch_save(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    torch.save(value, temporary)
    temporary.replace(path)


def _fingerprint(paths, texts, targets, config):
    digest = hashlib.sha256(json.dumps(config, sort_keys=True).encode())
    for path, text, target in zip(paths, texts, targets):
        digest.update(json.dumps([path.name, text, target.tolist()], ensure_ascii=False).encode("utf-8"))
    # 변경된 코드로 기존 fold가 조용히 재사용되는 것을 방지한다.
    for name in RUNTIME_FILES:
        path = Path(__file__).parent / name
        digest.update(path.name.encode())
        digest.update(path.read_bytes())
    return digest.hexdigest()


def train_pipeline(train_label_dir, output_dir, config=None):
    """Validation 인자를 받지 않는다. 보정 후에도 모델을 재학습하지 않는다."""
    supplied = config or {}
    unknown = set(supplied) - set(DEFAULT_CONFIG)
    if unknown:
        raise ValueError(f"Unknown config keys: {sorted(unknown)}")
    config = {**DEFAULT_CONFIG, **supplied}
    train_label_dir, output_dir = Path(train_label_dir).resolve(), Path(output_dir).resolve()
    if train_label_dir.name != "TL_서울_구급" or "Validation" in train_label_dir.parts:
        raise ValueError("학습 입력은 서울 Training의 TL_서울_구급 폴더여야 합니다.")
    if config["require_cuda"] and (config["device"] != "cuda" or not torch.cuda.is_available()):
        raise RuntimeError("전체 학습에는 GPU가 필요합니다. Colab GPU 런타임에서 실행하세요.")
    if config["device"] == "cuda":
        torch.set_float32_matmul_precision("high")
        print("GPU:", torch.cuda.get_device_name(0), flush=True)
    paths, texts, targets = load_dataset(train_label_dir, include_targets=True)
    dev, calibration, folds, groups = make_splits(
        texts, targets, config["calibration_size"], config["n_folds"], config["seed"],
    )
    output_dir.mkdir(parents=True, exist_ok=True)
    signature = _fingerprint(paths, texts, targets, config)
    run_path = output_dir / "run.json"
    if run_path.exists() and json.loads(run_path.read_text(encoding="utf-8"))["signature"] != signature:
        raise ValueError("출력 폴더에 다른 데이터/설정/코드의 실험이 있습니다. 새 output_dir를 지정하세요.")
    write_json(run_path, {"signature": signature, "config": config, "training_samples": len(paths)})
    write_json(output_dir / "splits.json", {
        "calibration": [paths[i].name for i in calibration],
        "folds": [{"fit": [paths[i].name for i in fit], "stop": [paths[i].name for i in stop]}
                  for fit, stop in folds],
        "seed": config["seed"], "grouping": "SHA256 of whitespace-normalized transcript only",
    })
    print(f"Training={len(paths):,}, development={len(dev):,}, calibration={len(calibration):,}", flush=True)
    tokenizer_dir = output_dir / "tokenizer"
    if (tokenizer_dir / "tokenizer_config.json").exists():
        tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True, local_files_only=True)
    else:
        tokenizer = AutoTokenizer.from_pretrained(config["model_name"], use_fast=True)
        tokenizer.save_pretrained(tokenizer_dir)
    documents = tokenize_dialogues(texts, tokenizer, config["max_length"], config["stride"])
    calibration_documents = [documents[i] for i in calibration]
    counts = np.asarray([len(document) for document in documents])
    print(f"대화 window 수: 평균={counts.mean():.2f}, 최대={counts.max()}, 긴 대화={int((counts > 1).sum()):,}", flush=True)
    neural_probs = np.zeros((len(calibration), len(TARGET_SYMPTOMS)), dtype=np.float64)
    fold_names, histories = [], []
    for number, (fit, stop) in enumerate(folds, 1):
        name = f"fold_{number}.ckpt"
        checkpoint_path, done_path = output_dir / name, output_dir / f"fold_{number}_done.json"
        print(f"\nFold {number}/{len(folds)}: fit={len(fit):,}, stop={len(stop):,}", flush=True)
        if done_path.exists() and checkpoint_path.exists():
            result = json.loads(done_path.read_text(encoding="utf-8"))
            print("완료된 fold 재사용 (동일 run signature 확인됨)", flush=True)
        else:
            result = train_model([documents[i] for i in fit], targets[fit],
                                 [documents[i] for i in stop], targets[stop], tokenizer,
                                 config, checkpoint_path, seed=config["seed"] + number * 1009)
            write_json(done_path, result)
        _release_memory()
        neural_probs += predict_checkpoint(checkpoint_path, calibration_documents, tokenizer, config)
        _release_memory()
        fold_names.append(name)
        histories.append(result)
    neural_probs /= len(folds)
    sparse_path = output_dir / "sparse.joblib"
    if sparse_path.exists():
        sparse = joblib.load(sparse_path)
    else:
        sparse = fit_sparse([texts[i] for i in dev], targets[dev], config)
        temporary = sparse_path.with_name("sparse.joblib.tmp")
        joblib.dump(sparse, temporary, compress=3)
        temporary.replace(sparse_path)
    sparse_probs = predict_sparse(sparse, [texts[i] for i in calibration])
    del sparse
    calibrator, report = calibrate_crossfit(targets[calibration], neural_probs, sparse_probs,
                                          groups[calibration], seed=config["seed"] + 2,
                                          n_splits=config["calibration_folds"])
    # OOF 확률을 final refit에 전용하지 않고, 실제 배포 ensemble의 확률을 보정한다.
    np.savez_compressed(output_dir / "calibration_predictions.npz", targets=targets[calibration],
                        neural=neural_probs, sparse=sparse_probs,
                        filenames=np.asarray([paths[i].name for i in calibration]))
    versions = {name: importlib.metadata.version(name) for name in
                ("torch", "transformers", "numpy", "scikit-learn", "iterative-stratification")}
    manifest = {"format": "mission3_ver4", "version": 1, "target_symptoms": list(TARGET_SYMPTOMS),
                "config": config, "folds": fold_names, "tokenizer": "tokenizer", "sparse": "sparse.joblib",
                "calibrator": calibrator, "signature": signature, "versions": versions}
    checkpoint_path = output_dir / "mission3_ver4_model.ckpt"
    _torch_save(checkpoint_path, manifest)
    report.update({"training_samples": len(paths), "development_samples": len(dev),
                   "model": f"{config['model_name']} {len(folds)}-fold ensemble + char/word TF-IDF LR",
                   "calibrator": calibrator, "fold_best_epochs": [x["best_epoch"] for x in histories],
                   "versions": versions, "validation_used_for_training_or_calibration": False})
    write_json(output_dir / "mission3_ver4_metrics.json", report)
    write_json(output_dir / "mission3_ver4_history.json", histories)
    # 노트북 하나만 Colab에 올려도 제출 코드와 실행 당시 설정을 남긴다.
    submission = output_dir / "submission"
    submission.mkdir(exist_ok=True)
    for name in (*RUNTIME_FILES, "requirements.txt", "README.md"):
        path = Path(__file__).parent / name
        if path.exists():
            shutil.copy2(path, submission / path.name)
    print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)
    print("Checkpoint:", checkpoint_path, flush=True)
    return checkpoint_path, report


def load_manifest(checkpoint_path):
    manifest = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    if manifest.get("format") != "mission3_ver4" or manifest.get("version") != 1:
        raise ValueError("지원하지 않는 checkpoint입니다. ver4의 manifest .ckpt를 지정하세요.")
    if manifest["target_symptoms"] != list(TARGET_SYMPTOMS) or not manifest["folds"]:
        raise ValueError("Checkpoint class order/folds mismatch")
    root = Path(checkpoint_path).resolve().parent
    for name in [*manifest["folds"], manifest["sparse"], manifest["tokenizer"]]:
        path = (root / name).resolve()
        if not path.is_relative_to(root) or not path.exists():
            raise FileNotFoundError(f"Checkpoint 구성 파일을 함께 보관해야 합니다: {name}")
    return manifest


def predict_pipeline(checkpoint_path, label_dir, output, device=None, batch_size=None):
    """저장된 encoder config/tokenizer만 사용하므로 학습 후 오프라인 추론 가능."""
    checkpoint_path = Path(checkpoint_path).resolve()
    manifest = load_manifest(checkpoint_path)
    root = checkpoint_path.parent
    config = dict(manifest["config"])
    config["device"] = device or ("cuda" if torch.cuda.is_available() else "cpu")
    if batch_size is not None:
        config["inference_batch_size"] = batch_size
    paths, texts, _ = load_dataset(Path(label_dir), include_targets=False)
    tokenizer = AutoTokenizer.from_pretrained(root / manifest["tokenizer"], use_fast=True, local_files_only=True)
    documents = tokenize_dialogues(texts, tokenizer, config["max_length"], config["stride"])
    neural_probs = np.zeros((len(paths), len(TARGET_SYMPTOMS)), dtype=np.float64)
    for name in manifest["folds"]:
        print("Predict:", name, flush=True)
        neural_probs += predict_checkpoint(root / name, documents, tokenizer, config)
        _release_memory()
    neural_probs /= len(manifest["folds"])
    sparse = joblib.load(root / manifest["sparse"])
    sparse_probs = predict_sparse(sparse, texts)
    probabilities = apply_calibrator(neural_probs, sparse_probs, manifest["calibrator"])
    predictions = write_predictions(paths, probabilities, manifest["calibrator"]["thresholds"], Path(output))
    print(f"{len(paths):,} predictions saved: {output}", flush=True)
    return paths, probabilities, predictions


def evaluate_csv(label_dir, prediction_path):
    """이미 저장된 예측을 파일명으로 정답에 연결한다. 모델/임계값 변경 없음."""
    with Path(prediction_path).open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != ["label file name", "symptom"]:
            raise ValueError("CSV header must be: label file name,symptom")
        rows = list(reader)
    by_name = {}
    for row in rows:
        if row["label file name"] in by_name:
            raise ValueError("Duplicate prediction filename")
        labels = ast.literal_eval(row["symptom"])
        if not isinstance(labels, list) or any(x not in TARGET_SYMPTOMS for x in labels) or len(set(labels)) != len(labels):
            raise ValueError("CSV symptom must be a list of unique target strings")
        by_name[row["label file name"]] = [int(x in labels) for x in TARGET_SYMPTOMS]
    paths, _, targets = load_dataset(Path(label_dir), include_targets=True)
    if set(by_name) != {path.name for path in paths}:
        raise ValueError("CSV와 JSON 파일명 집합이 일치하지 않습니다.")
    return score_predictions(targets, np.asarray([by_name[path.name] for path in paths], dtype=np.int8))


In [ ]:
%%writefile train.py
"""Run all Training-only fitting and calibration; never accepts Validation."""
import argparse
import json
from pathlib import Path

if __package__:
    from .pipeline import train_pipeline
else:
    from pipeline import train_pipeline


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--label_dir", required=True, type=Path)
    parser.add_argument("--output_dir", required=True, type=Path)
    parser.add_argument("--config", type=Path, help="Optional JSON overrides of DEFAULT_CONFIG")
    args = parser.parse_args()
    config = json.loads(args.config.read_text(encoding="utf-8")) if args.config else None
    train_pipeline(args.label_dir, args.output_dir, config)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile inference.py
"""Official one-command inference. Mission 3 accepts but does not read audio_dir."""
import argparse
from pathlib import Path

if __package__:
    from .pipeline import predict_pipeline
else:
    from pipeline import predict_pipeline


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--audio_dir", required=True, help="Organizer interface; unused text-only task")
    parser.add_argument("--label_dir", required=True, type=Path)
    parser.add_argument("--ckpt_path", required=True, type=Path)
    parser.add_argument("--output", required=True, type=Path)
    parser.add_argument("--device", choices=["cpu", "cuda"])
    parser.add_argument("--batch_size", type=int)
    args = parser.parse_args()
    predict_pipeline(args.ckpt_path, args.label_dir, args.output, args.device, args.batch_size)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate.py
"""Score frozen CSV predictions after inference, without changing the model."""
import argparse
import json
from pathlib import Path

if __package__:
    from .data import write_json
    from .pipeline import evaluate_csv
else:
    from data import write_json
    from pipeline import evaluate_csv


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--label_dir", required=True, type=Path)
    parser.add_argument("--prediction", required=True, type=Path)
    parser.add_argument("--output", type=Path)
    args = parser.parse_args()
    report = evaluate_csv(args.label_dir, args.prediction)
    if args.output:
        write_json(args.output, report)
    print(json.dumps(report, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
Path('requirements.txt').write_text('torch>=2.6,<3\ntransformers==4.57.1\nnumpy>=1.26,<3\nscipy>=1.11,<2\nscikit-learn>=1.4,<2\niterative-stratification==0.1.9\njoblib>=1.3,<2\n', encoding='utf-8')


In [ ]:
Path('README.md').write_text('# Mission 3 ver4 실행 안내\n\n실행 파일은 [src/mission3_ver4.ipynb](../mission3_ver4.ipynb)입니다. 이 노트북 하나만 Colab에 올리면 필요한 모델·학습·추론 코드를 생성합니다. 이 폴더의 `.py` 파일은 같은 기능을 로컬이나 제출 환경에서 실행하기 위한 코드입니다.\n\n## 기존 결과와 개선 근거\n\n| 실행 버전 | 실제 Validation Macro F1 | 근거 |\n|---|---:|---|\n| `mission3_all_in_one_ver2` → results/mission3/ver1 | 0.600123 | `results/mission3/ver1/mission3_metrics.json` |\n| `mission3_ver3` → results/mission3/ver3 | 0.626015 | `results/mission3/ver3/mission3_ver3_metrics.json` |\n| ver4 | 아직 미측정 | 전체 GPU 학습 후 평가 필요 |\n\nver3의 Training 내부 F1은 3 epoch에서 0.644361로 최고였고, 15 epoch에는 0.601536으로 하락했습니다. 학습 손실은 계속 감소해 과적합 징후가 있습니다. ver4는 별도 stopping 집합의 Macro AP로 최적 epoch를 선택하고 조기 종료합니다. AP는 임계값 없이 예측 순위를 평가하며, 최종 대회 지표는 그대로 **9개 증상의 Macro F1**입니다.\n\nTraining 29,200개를 저장된 ver3 tokenizer로 직접 측정했을 때 384토큰을 넘는 대화는 3,292개(11.27%), 512토큰을 넘는 대화는 788개(2.70%)였습니다. 특수 토큰을 포함한 최대 길이는 873토큰입니다. ver4는 긴 대화의 뒷부분도 처리하도록 설계했습니다.\n\n**현재 작성 환경에는 GPU가 없어 ver4 전체 학습과 Validation 성능은 확인하지 못했습니다.** 작은 CPU 모델을 이용한 실행 검증은 코드 경로 확인용입니다. 기존 버전보다 성능이 높다는 판단은 전체 학습·추론이 끝난 뒤 실제 점수로 해야 합니다.\n\n## 모델과 데이터 분리\n\n- `klue/roberta-large`를 5개 fold에서 각각 학습하고 예측 확률을 평균냅니다.\n- 한 window는 최대 512토큰이며 인접 window는 64토큰이 겹칩니다. 마지막 window까지 유지하고, 증상별 attention으로 대화 전체를 모은 뒤 **통화당 한 번** BCE 손실을 계산합니다.\n- 문자 2–5그램과 단어 1–2그램 TF-IDF + 증상별 Logistic Regression을 함께 학습합니다.\n- 증상마다 신경망/TF-IDF의 혼합 비율과 판정 임계값을 Training 내부에서 결정합니다. 어떤 통화도 최소 1개 증상을 출력하도록 강제하지 않습니다.\n\n서울 Training을 약 90% development와 10% calibration으로 먼저 나눕니다. Development 내부 5-fold는 원래 Training 기준으로 각 fold마다 약 72% fit, 18% stopping입니다. 동일한 정규화 대화는 같은 그룹에 두어 분할을 넘지 않게 합니다.\n\nCalibration 10%는 신경망 학습, TF-IDF 학습, epoch 선택에 전혀 사용하지 않습니다. TF-IDF는 development 90%에만 fit합니다. 완성된 5개 신경망의 평균 확률과 TF-IDF 확률을 calibration에 적용한 뒤, 그 안에서 다시 5-fold 교차 보정하여 혼합 비율·임계값 선택의 일반화 점수를 계산합니다. 배포용 보정값은 calibration 전체로 결정합니다. **보정 후 기본 모델을 다시 학습하지 않고 같은 모델들을 그대로 추론에 사용합니다.** 공식 Validation은 이 과정에 들어가지 않습니다.\n\n주요 기본값은 `pipeline.py`의 `DEFAULT_CONFIG`에 있습니다.\n\n| 설정 | 기본값 |\n|---|---|\n| 모델 / fold | `klue/roberta-large` / 5 |\n| 최대 epoch / 조기 종료 patience | 6 / 2 |\n| 통화 batch / gradient accumulation | 2 / 16 |\n| encoder / head learning rate | `1.5e-5` / `1e-4` |\n| 문자 / 단어 최대 특징 수 | 120,000 / 40,000 |\n\n## Colab에서 실행\n\n1. [mission3_ver4.ipynb](../mission3_ver4.ipynb)를 Colab에 업로드합니다.\n2. GPU 런타임을 선택합니다. 기본 large 모델의 5-fold 학습에는 **A100 또는 L4를 권장**합니다. 사용 가능한 장치에 따라 시간과 메모리 사용량은 달라집니다.\n3. 제공 데이터의 압축을 풀고 `TL_서울_구급`, `VL_서울_구급` 폴더 이름을 유지합니다. Mission 3은 JSON만 읽습니다.\n4. 위에서부터 모든 셀을 실행하고 Google Drive를 연결합니다. 노트북은 `/content`를 먼저 탐색하고 Drive의 알려진 데이터 경로를 확인합니다. 자동 탐색에 실패하면 노트북의 데이터 경로 설정에 실제 폴더를 지정합니다.\n5. 학습·보정이 끝난 뒤 Validation 추론과 평가 셀을 실행합니다. 실행 로그가 남은 노트북도 저장합니다.\n\n기본 결과 폴더는 Colab에서 `/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver4`, 로컬에서 프로젝트의 `results/mission3/ver4`입니다. 신경망 가중치 5개를 저장하므로 Drive 여유 공간도 확인합니다. 패키지와 최초 공개 pretrained 모델 다운로드에는 인터넷 연결이 필요하며, 학습 완료 후에는 저장된 구성으로 오프라인 추론할 수 있습니다.\n\n## 로컬 또는 제출 환경에서 실행\n\n다음 명령은 `DCC_AIdeas` 프로젝트 루트 기준입니다. 전체 학습 명령은 CUDA GPU를 요구합니다.\n\n```bash\npython -m pip install -r src/mission3_ver4/requirements.txt\n\npython src/mission3_ver4/train.py \\\n  --label_dir "DATA/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급" \\\n  --output_dir results/mission3/ver4\n\npython src/mission3_ver4/inference.py \\\n  --audio_dir "DATA/대학부 데이터/Validation/1.원천데이터/VS_서울_구급" \\\n  --label_dir "DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급" \\\n  --ckpt_path results/mission3/ver4/mission3_ver4_model.ckpt \\\n  --output results/mission3/ver4/mission3.csv\n\npython src/mission3_ver4/evaluate.py \\\n  --label_dir "DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급" \\\n  --prediction results/mission3/ver4/mission3.csv \\\n  --output results/mission3/ver4/validation_metrics.json\n```\n\n`--audio_dir`는 주최 측 인터페이스를 맞추기 위해 받으며 음성 파일을 읽지 않습니다. 추론은 `--device cpu`도 지원하지만 large 모델 5개를 처리하므로 GPU보다 느릴 수 있습니다. 학습 설정은 `train.py --config 설정.json`으로 변경할 수 있습니다. 설정을 바꿀 때는 새 `--output_dir`를 사용합니다.\n\n## 결과 읽기와 파일 보관\n\n| 파일 | 의미 |\n|---|---|\n| `mission3_ver4_metrics.json` | Training 내부 보정 점수, 증상별 교차 보정 F1, 최종 보정값 |\n| `mission3_ver4_history.json` | 각 fold의 epoch별 학습·stopping 기록 |\n| `validation_metrics.json` | 위 평가 명령으로 만든 공식 Validation Macro F1와 증상별 점수 |\n| `mission3.csv` | `label file name,symptom` 형식의 예측 결과 |\n| `run.json`, `splits.json` | 실행 설정·데이터/코드 식별값과 분할 파일 목록 |\n| `submission/` | 실행 당시 `.py` 파일, requirements와 실행 안내 사본 |\n\n`calibration_crossfit_macro_f1`는 각 보정 평가 샘플의 정답을 보정값 선택에서 제외하여 계산한 **Training 내부 점수**입니다. `calibration_fit_macro_f1`는 같은 calibration 데이터에서 보정값을 정하고 다시 평가한 점수라 낙관적일 수 있습니다. 둘을 Validation 점수로 보고하면 안 됩니다. 공식 Validation 평가를 모델 선택이나 임계값 재조정에 반복 사용하지 않습니다.\n\n**`mission3_ver4_model.ckpt`는 구성 목록과 보정값을 담은 manifest입니다. 이 파일 하나만 옮기면 추론할 수 없습니다.** 결과 디렉터리 전체를 함께 보관하거나 최소한 다음 구성을 유지합니다.\n\n```text\nver4/\n├─ mission3_ver4_model.ckpt\n├─ fold_1.ckpt ... fold_5.ckpt\n├─ tokenizer/\n├─ sparse.joblib\n└─ submission/\n```\n\nCSV의 symptom 값은 기존 결과와 같은 문자열 목록 표현입니다. 예를 들어 `[]`, `[\'구토\', \'복통\']`처럼 저장하며 CSV 인용 처리는 코드가 담당합니다. 주최 측 sample submission에서 더 구체적인 형식을 제공하면 해당 규격도 확인합니다.\n\n## 중단 후 다시 실행\n\n같은 데이터·설정·코드와 같은 결과 폴더로 다시 실행하면 `fold_N.ckpt`와 `fold_N_done.json`이 모두 존재하는 **완료된 fold만 재사용**합니다. 실행 도중 끊긴 fold는 처음부터 다시 학습합니다. 마지막 미니배치나 optimizer 상태에서 이어가는 기능은 아닙니다.\n\n데이터·설정·코드의 식별값이 기존 실행과 다르면 혼합된 결과를 만들지 않도록 중단합니다. 이 경우 새 결과 폴더를 지정합니다. 메모리가 부족하면 통화 batch와 window batch를 줄이는 설정을 검토하고 새 폴더에서 시작합니다.\n\n## 대회 조건과 참고 자료\n\n판단 기준은 [docs의 출제 PDF](../../docs/0820_DCC%20대학부_예선_출제문제.pdf) 5쪽, 9–12쪽입니다. 서울 Training만 학습하고 양측의 전사 `text`만 입력으로 사용합니다. `speaker`, 시간, 주소, 성별, 정답 라벨은 입력 특징에서 제외합니다. 대상은 `고열, 구토, 두통, 복통, 어지러움, 열상, 오심, 전신쇠약, 호흡곤란`이며, 나머지 라벨만 제거하고 통화는 유지합니다. 외부 추가 데이터와 상용 AI API를 사용하지 않습니다.\n\n최종 제출에는 추론 코드, 학습 로그 포함 노트북, 학습 완료 가중치, 모델 코드, requirements 및 결과 정리 PPT가 필요합니다. 이 구현의 학습을 완료하고 로그·가중치를 보관해야 하며 PPT는 별도로 준비합니다.\n\n공식 구현 참고: [KLUE RoBERTa large](https://huggingface.co/klue/roberta-large), [Transformers tokenizer](https://huggingface.co/docs/transformers/en/main_classes/tokenizer), [PyTorch AMP](https://docs.pytorch.org/docs/stable/notes/amp_examples.html). 모델 로딩·토큰 처리·혼합 정밀도 API 참고 자료이며 Mission 3의 성능 향상을 보장하는 근거는 아닙니다.\n', encoding='utf-8')


In [ ]:
# 5. 성능 중심 기본 설정. 먼저 기본값으로 한 번 실행해 비교하세요.
import json
import torch
from pipeline import DEFAULT_CONFIG, train_pipeline, predict_pipeline, evaluate_csv
from data import TARGET_SYMPTOMS, write_json

CONFIG = {**DEFAULT_CONFIG,
          "model_name": "klue/roberta-large",
          "n_folds": 5,
          "max_epochs": 6,
          "patience": 2,
          "batch_size": 2,
          "gradient_accumulation_steps": 16,
          "window_batch_size": 2}
if not torch.cuda.is_available():
    raise RuntimeError("전체 학습에는 GPU가 필요합니다. Colab 런타임 유형을 GPU로 바꾸세요.")
print("GPU:", torch.cuda.get_device_name(0))
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))


## 6. 학습 및 Training 내부 보정

각 fold에서 학습 중 가장 좋은 **stop Macro AP**의 가중치를 보관하고 조기 종료합니다.
fold 다섯 개를 평균낸 보정 집합 예측과 TF-IDF 예측으로 증상별 가중치·임계값을 정합니다.
`calibration_fit_macro_f1`은 보정에 사용한 점수이고, `calibration_crossfit_macro_f1`은 보정 집합 내부 교차 평가입니다.
둘 다 공식 Validation 점수가 아닙니다.

런타임이 끊겼다면 같은 데이터·설정·코드로 다시 실행하면 완료된 fold를 재사용합니다.
진행 중이던 fold는 처음부터 재학습합니다. 설정을 바꾼 실험은 2번 셀의 `OUTPUT_DIR`를 새 폴더로 바꾸세요.


In [ ]:
CHECKPOINT_PATH, training_report = train_pipeline(TRAIN_LABEL_DIR, OUTPUT_DIR, CONFIG)
print("Training 내부 보정 교차평가:", training_report["calibration_crossfit_macro_f1"])


## 7. 고정된 모델로 Validation CSV 생성

학습·보정이 모두 끝난 체크포인트를 다시 읽습니다. 이 단계에서는 Validation의 `text`만 추론 입력으로 사용합니다.


In [ ]:
PREDICTION_PATH = OUTPUT_DIR / "mission3.csv"
if VALID_LABEL_DIR is not None:
    _, _, validation_predictions = predict_pipeline(CHECKPOINT_PATH, VALID_LABEL_DIR, PREDICTION_PATH)
else:
    print("Validation 폴더가 없어 CSV 생성을 생략했습니다. 경로 지정 후 이 셀부터 실행하세요.")


## 8. 마지막 평가와 제출 파일 확인

CSV를 먼저 저장한 다음에만 Validation 정답을 읽어 평가합니다. 이 점수에 맞추어 임계값을 다시 조정하지 않습니다.
학습 로그가 남은 이 노트북도 저장하세요. PDF는 별도 결과 정리 PPT도 요구합니다.


In [ ]:
if VALID_LABEL_DIR is not None:
    validation_report = evaluate_csv(VALID_LABEL_DIR, PREDICTION_PATH)
    write_json(OUTPUT_DIR / "validation_metrics.json", validation_report)
    metrics_path = OUTPUT_DIR / "mission3_ver4_metrics.json"
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    metrics["validation_macro_f1"] = validation_report["macro_f1"]
    metrics["validation"] = validation_report
    write_json(metrics_path, metrics)
    print(f"ver1: 0.600123 | ver3: 0.626015 | ver4: {validation_report['macro_f1']:.6f}")
    for symptom in TARGET_SYMPTOMS:
        print(symptom, validation_report["per_symptom"][symptom])

print("결과 폴더 전체를 보관하세요:", OUTPUT_DIR)
print("manifest:", CHECKPOINT_PATH)
print("실행 코드:", OUTPUT_DIR / "submission")


최종 추론 명령은 다음과 같습니다. `.ckpt` manifest 하나만 복사하지 말고
**fold_*.ckpt, tokenizer/, sparse.joblib**를 같은 결과 폴더에 함께 보관해야 합니다.

```bash
python submission/inference.py --audio_dir "WAV 폴더" --label_dir "JSON 폴더" \
  --ckpt_path mission3_ver4_model.ckpt --output mission3.csv
```

모델 구현 근거: [KLUE RoBERTa](https://huggingface.co/klue/roberta-large),
[tokenizer overflow](https://huggingface.co/docs/transformers/en/main_classes/tokenizer),
[PyTorch AMP](https://docs.pytorch.org/docs/stable/notes/amp_examples.html).
실제 점수 향상 여부는 위 학습·최종 평가 출력으로 확인합니다.
